In [1]:
import os
import bitsandbytes
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model
from lowresource_llm_evaluation import benchmark
from lowresource_llm_evaluation.interferenciaLinguistica import loadLexicon
import pandas as pd
import numpy as np
import torch
from huggingface_hub import login
import time
import json
import gc
from dotenv import load_dotenv

base = "./"
load_dotenv(base + "secrets.env")
login(token=os.getenv("HF_TOKEN"))

2026-05-11 19:40:10.274335: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-05-11 19:40:10.274477: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-05-11 19:40:10.383012: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-05-11 19:40:10.591249: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-05-11 19:40:12.409764: W tensorflow/compiler/tf2

Token will not been saved to git credential helper. Pass `add_to_git_credential=True` if you want to set the git credential as well.
Token is valid (permission: read).
Your token has been saved to /root/.cache/huggingface/token
Login successful


In [ ]:
def clean_graphics_card():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
    gc.collect()
    torch.cuda.empty_cache()

def load_gallego():
    with open(base + "EvalDatasets/Raw/idioms_train_es.txt", "r", encoding="utf-8") as fEsp:
        esp = fEsp.readlines()

    with open(base + "EvalDatasets/Raw/idioms_train_gl.txt", "r", encoding="utf-8") as fGl:
        gl = fGl.readlines()

    with open(base + "EvalDatasets/Raw/idioms_test_es.txt", "r", encoding="utf-8") as fEsp:
        espTest = fEsp.readlines()

    with open(base + "EvalDatasets/Raw/idioms_test_gl.txt", "r", encoding="utf-8") as fGl:
        glTest = fGl.readlines()
    return pd.DataFrame(np.array((esp + espTest, gl + glTest)).T, columns=["es","gl"])

def split_for_translation_and_roundtrip(df, N):
    total = len(df)

    # Caso 1: hay al menos 2N → no hay solape
    if total >= 2 * N:
        df_trans = df.iloc[:N]
        df_round = df.iloc[N:2*N]
        return df_trans, df_round

    # Caso 2: no hay suficientes → roundtrip desde el final hacia atrás
    df_trans = df.iloc[:N]

    # Seleccionamos los últimos N sin tocar los primeros N
    df_round = df.iloc[-N:]

    return df_trans, df_round


def evaluate_benchmark(model_name, idioma, token, N= 20, device="cuda", debug=False, remote_code=True):
    
    # 1. Define the 4-bit quantization configuration
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=  torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )

    # 2. Load the tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=remote_code)

    # 3. Load the pre-trained language model with quantization
    model = AutoModelForCausalLM.from_pretrained(
                model_name,
                quantization_config=bnb_config,
                trust_remote_code=remote_code,
                tie_word_embeddings=False, # Added to silence the warning about tied weights
                token = token,
                device_map="auto"
            )
    
    tokenizer.pad_token = tokenizer.eos_token
    model.config.pad_token_id = tokenizer.eos_token_id


    print(f"Tokenizer loaded: {tokenizer.__class__.__name__}")
    print(f"Model loaded with 4-bit quantization: {model.__class__.__name__}")
    print(f"Model device: {model.device}")
    codigos = {"aranes": "aran" , 
               "asturiano": "ast", 
               "gallego": "gl"}
    textos = {"aranes": pd.read_parquet("hf://datasets/projecte-aina/ES-OC_Parallel_Corpus/es-arn_corpus.parquet") , 
            "asturiano": pd.read_parquet("hf://datasets/projecte-aina/ES-AST_Parallel_Corpus/es-ast_corpus.parquet"), 
            "gallego": load_gallego()}
    
    df_textos, df_textos_round = split_for_translation_and_roundtrip(textos[idioma],N)
    del textos # Limpiamos RAM
    results = benchmark(model, tokenizer, 
            df_textos = df_textos,
            list_textos_round = df_textos_round[df_textos_round.columns[1]],
            lang_eval= codigos[idioma],
            df_huecos=  pd.read_csv(base + f"EvalDatasets/Huecos/{idioma}.csv").head(N),
            df_anotado = pd.read_csv(base + f"EvalDatasets/Anotado/{idioma}.csv").head(N),
            lexicon_target = loadLexicon(base + f"lexicons/{codigos[idioma]}.txt"),
            lexicons_comparison = {"es": loadLexicon(base + f"lexicons/es.txt"), "fr": loadLexicon(base + f"lexicons/fr.txt")},
            roundtrip_langs= ["es"],
            cortar_ortografico = True,
            cortar_vocabulario = True,
            debug=debug)
    # Lberamos GPU
    try:
        model.to("cpu")
        del model
        del tokenizer
        clean_graphics_card()
    except Exception as e:
        print("Borrar el modelo ha fallado")
        print(e)
    return results


In [ ]:
import torch
print(torch.cuda.is_available())
import transformers
print(transformers.__version__)

True
4.40.2


# Aranés

## Mistral 7B 

In [4]:
idioma = "aranes"
modelo = "mistralai/Mistral-7B-Instruct-v0.3"
resultados = evaluate_benchmark(modelo, idioma, os.getenv("HF_TOKEN"), 100)
date = time.localtime(time.time())
filename = f"resultados_{idioma}_{modelo.split('/')[-1]}_{time.strftime('%m-%d_%H-%M-%S', date)}.json"
with open(base + filename, "w") as f:
    json.dump(resultados, f)

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

You set `add_prefix_space`. The tokenizer needs to be converted from the slow tokenizers


...

TRADUCCIÓN ROUND TRIP acabado  en 20.79
Empezando VOCABULARIO


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for o

VOCABULARIO acabado  en 6.83
Empezando ORTOGRAFÍA


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for o

ORTOGRAFÍA acabado en 13.16

                        Evaluación de Calidad de Lengua                         

+-----------------+-------------------------------------------------------+
| Clave           | Valor                                                 |
+-----------------+-------------------------------------------------------+
| ttr             | 0.4681802796379446                                    |
| entropy         | 6.8886785153068555                                    |
| ngram_overlap   | 0.0                                                   |
| freq_target     | 0.4698188801536519                                    |
| freq_comparison | {'es': 0.4670386513003484, 'fr': 0.30053548526423357} |
| calidad         | 0.17817821296635833                                   |
+-----------------+-------------------------------------------------------+

                            Evaluación de Traducción                            

+------+--------------------+
| Clave | Valor 

## Salamandra

In [5]:
idioma = "aranes"
modelo = "BSC-LT/salamandra-7b-instruct"
resultados = evaluate_benchmark(modelo, idioma, os.getenv("HF_TOKEN"), 100)
date = time.localtime(time.time())
filename = f"resultados_{idioma}_{modelo.split('/')[-1]}_{time.strftime('%m-%d_%H-%M-%S', date)}.json"
with open(base + filename, "w") as f:
    json.dump(resultados, f)

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/4.81M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/19.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/513 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/730 [00:00<?, ?B/s]

...

TRADUCCIÓN ROUND TRIP acabado  en 6.03
Empezando VOCABULARIO


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for o

VOCABULARIO acabado  en 1.42
Empezando ORTOGRAFÍA


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for o

ORTOGRAFÍA acabado en 4.61

                        Evaluación de Calidad de Lengua                         

+-----------------+-------------------------------------------------------+
| Clave           | Valor                                                 |
+-----------------+-------------------------------------------------------+
| ttr             | 0.7193543859948686                                    |
| entropy         | 7.426086390806821                                     |
| ngram_overlap   | 0.0                                                   |
| freq_target     | 0.39847381921641195                                   |
| freq_comparison | {'es': 0.44849532749550347, 'fr': 0.3955292337123576} |
| calidad         | 0.22042726249616806                                   |
+-----------------+-------------------------------------------------------+

                            Evaluación de Traducción                            

+------+--------------------+
| Clave | Valor  

## Gemma

In [6]:
idioma = "aranes"
modelo = "google/gemma-7b-it"
resultados = evaluate_benchmark(modelo, idioma, os.getenv("HF_TOKEN"), 100)
date = time.localtime(time.time())
filename = f"resultados_{idioma}_{modelo.split('/')[-1]}_{time.strftime('%m-%d_%H-%M-%S', date)}.json"
with open(base + filename, "w") as f:
    json.dump(resultados, f)

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

...

Some weights of GemmaForCausalLM were not initialized from the model checkpoint at google/gemma-7b-it and are newly initialized: ['lm_head.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

Tokenizer loaded: GemmaTokenizerFast
Model loaded with 4-bit quantization: GemmaForCausalLM
Model device: cuda:0
Empezando CALIDAD DE LENGUA
CALIDAD DE LENGUA acabada en 5.45 minutos
Empezando TRADUCCIÓN DIRECTA
TRADUCCIÓN DIRECTA acabada en 14.61
Empezando TRADUCCIÓN ROUND TRIP
TRADUCCIÓN ROUND TRIP acabado  en 29.69
Empezando VOCABULARIO
VOCABULARIO acabado  en 7.75
Empezando ORTOGRAFÍA


You shouldn't move a model that is dispatched using accelerate hooks.


ORTOGRAFÍA acabado en 14.63

                        Evaluación de Calidad de Lengua                         

+-----------------+--------------------------------------------------------+
| Clave           | Valor                                                  |
+-----------------+--------------------------------------------------------+
| ttr             | 0.22215230360923108                                    |
| entropy         | 4.423389496421388                                      |
| ngram_overlap   | 0.0                                                    |
| freq_target     | 0.012912150804994794                                   |
| freq_comparison | {'es': 0.01987299513585628, 'fr': 0.16334126469867077} |
| calidad         | 0.026226949598649973                                   |
+-----------------+--------------------------------------------------------+

                            Evaluación de Traducción                            

+------+---------------------+
| Cla

## Qwen

In [7]:
idioma = "aranes"
modelo = "Qwen/Qwen2.5-7B-Instruct"
resultados = evaluate_benchmark(modelo, idioma, os.getenv("HF_TOKEN"), 100)
date = time.localtime(time.time())
filename = f"resultados_{idioma}_{modelo.split('/')[-1]}_{time.strftime('%m-%d_%H-%M-%S', date)}.json"
with open(base + filename, "w") as f:
    json.dump(resultados, f)

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


...

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Tokenizer loaded: Qwen2TokenizerFast
Model loaded with 4-bit quantization: Qwen2ForCausalLM
Model device: cuda:0
Empezando CALIDAD DE LENGUA
CALIDAD DE LENGUA acabada en 3.5 minutos
Empezando TRADUCCIÓN DIRECTA
TRADUCCIÓN DIRECTA acabada en 5.51
Empezando TRADUCCIÓN ROUND TRIP
TRADUCCIÓN ROUND TRIP acabado  en 20.18
Empezando VOCABULARIO
VOCABULARIO acabado  en 7.0
Empezando ORTOGRAFÍA


You shouldn't move a model that is dispatched using accelerate hooks.


ORTOGRAFÍA acabado en 13.67

                        Evaluación de Calidad de Lengua                         

+-----------------+-------------------------------------------------------+
| Clave           | Valor                                                 |
+-----------------+-------------------------------------------------------+
| ttr             | 0.41510439408343125                                   |
| entropy         | 6.345908546382653                                     |
| ngram_overlap   | 0.0                                                   |
| freq_target     | 0.4592811177721702                                    |
| freq_comparison | {'es': 0.5093525428531387, 'fr': 0.30278169911870917} |
| calidad         | 0.15105885581294806                                   |
+-----------------+-------------------------------------------------------+

                            Evaluación de Traducción                            

+------+--------------------+
| Clave | Valor 

# Asturiano

## Mistral 7B 

In [8]:
idioma = "asturiano"
modelo = "mistralai/Mistral-7B-Instruct-v0.3"
resultados = evaluate_benchmark(modelo, idioma, os.getenv("HF_TOKEN"), 100)
date = time.localtime(time.time())
filename = f"resultados_{idioma}_{modelo.split('/')[-1]}_{time.strftime('%m-%d_%H-%M-%S', date)}.json"
with open(base + filename, "w") as f:
    json.dump(resultados, f)

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Tokenizer loaded: LlamaTokenizerFast
Model loaded with 4-bit quantization: MistralForCausalLM
Model device: cuda:0


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Empezando CALIDAD DE LENGUA


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


...

TRADUCCIÓN ROUND TRIP acabado  en 21.11
Empezando VOCABULARIO


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for o

VOCABULARIO acabado  en 7.14
Empezando ORTOGRAFÍA


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for o

ORTOGRAFÍA acabado en 10.68

                        Evaluación de Calidad de Lengua                         

+-----------------+-------------------------------------------------------+
| Clave           | Valor                                                 |
+-----------------+-------------------------------------------------------+
| ttr             | 0.4699091544976808                                    |
| entropy         | 6.962593199121493                                     |
| ngram_overlap   | 0.0003952569169960474                                 |
| freq_target     | 0.5067098567203214                                    |
| freq_comparison | {'es': 0.7360411168472812, 'fr': 0.33464878857178904} |
| calidad         | 0.15457230109746392                                   |
+-----------------+-------------------------------------------------------+

                            Evaluación de Traducción                            

+------+-------------------+
| Clave | Valor  

## Salamandra

In [9]:
idioma = "asturiano"
modelo = "BSC-LT/salamandra-7b-instruct"
resultados = evaluate_benchmark(modelo, idioma, os.getenv("HF_TOKEN"), 100)
date = time.localtime(time.time())
filename = f"resultados_{idioma}_{modelo.split('/')[-1]}_{time.strftime('%m-%d_%H-%M-%S', date)}.json"
with open(base + filename, "w") as f:
    json.dump(resultados, f)

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Tokenizer loaded: LlamaTokenizerFast
Model loaded with 4-bit quantization: LlamaForCausalLM
Model device: cuda:0


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Empezando CALIDAD DE LENGUA


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


...

TRADUCCIÓN ROUND TRIP acabado  en 25.41
Empezando VOCABULARIO


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for o

VOCABULARIO acabado  en 3.95
Empezando ORTOGRAFÍA


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for o

ORTOGRAFÍA acabado en 13.74

                        Evaluación de Calidad de Lengua                         

+-----------------+------------------------------------------------------+
| Clave           | Valor                                                |
+-----------------+------------------------------------------------------+
| ttr             | 0.7616744188047437                                   |
| entropy         | 7.576134957968035                                    |
| ngram_overlap   | 0.0                                                  |
| freq_target     | 0.5515114043679795                                   |
| freq_comparison | {'es': 0.891240945720505, 'fr': 0.26023705633757943} |
| calidad         | 0.20998974011852206                                  |
+-----------------+------------------------------------------------------+

                            Evaluación de Traducción                            

+------+--------------------+
| Clave | Valor           

## Gemma

In [4]:
idioma = "asturiano"
modelo = "google/gemma-7b-it"
resultados = evaluate_benchmark(modelo, idioma, os.getenv("HF_TOKEN"), 100)
date = time.localtime(time.time())
filename = f"resultados_{idioma}_{modelo.split('/')[-1]}_{time.strftime('%m-%d_%H-%M-%S', date)}.json"
with open(base + filename, "w") as f:
    json.dump(resultados, f)

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

...

Some weights of GemmaForCausalLM were not initialized from the model checkpoint at google/gemma-7b-it and are newly initialized: ['lm_head.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

Tokenizer loaded: GemmaTokenizerFast
Model loaded with 4-bit quantization: GemmaForCausalLM
Model device: cuda:0
Empezando CALIDAD DE LENGUA
CALIDAD DE LENGUA acabada en 6.01 minutos
Empezando TRADUCCIÓN DIRECTA
TRADUCCIÓN DIRECTA acabada en 15.8
Empezando TRADUCCIÓN ROUND TRIP
TRADUCCIÓN ROUND TRIP acabado  en 32.12
Empezando VOCABULARIO
VOCABULARIO acabado  en 8.02
Empezando ORTOGRAFÍA


You shouldn't move a model that is dispatched using accelerate hooks.


ORTOGRAFÍA acabado en 15.21

                        Evaluación de Calidad de Lengua                         

+-----------------+----------------------------------------------------------+
| Clave           | Valor                                                    |
+-----------------+----------------------------------------------------------+
| ttr             | 0.39823963970686005                                      |
| entropy         | 6.354675375785591                                        |
| ngram_overlap   | 0.0                                                      |
| freq_target     | 0.04556475041421569                                      |
| freq_comparison | {'fr': 0.036028139923717034, 'es': 0.049166213446924324} |
| calidad         | 0.10792830685987043                                      |
+-----------------+----------------------------------------------------------+

                            Evaluación de Traducción                            

+------+--------

## Qwen

In [5]:
idioma = "asturiano"
modelo = "Qwen/Qwen2.5-7B-Instruct"
resultados = evaluate_benchmark(modelo, idioma, os.getenv("HF_TOKEN"), 100)
date = time.localtime(time.time())
filename = f"resultados_{idioma}_{modelo.split('/')[-1]}_{time.strftime('%m-%d_%H-%M-%S', date)}.json"
with open(base + filename, "w") as f:
    json.dump(resultados, f)

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


...

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Tokenizer loaded: Qwen2TokenizerFast
Model loaded with 4-bit quantization: Qwen2ForCausalLM
Model device: cuda:0
Empezando CALIDAD DE LENGUA
CALIDAD DE LENGUA acabada en 3.1 minutos
Empezando TRADUCCIÓN DIRECTA
TRADUCCIÓN DIRECTA acabada en 8.86
Empezando TRADUCCIÓN ROUND TRIP
TRADUCCIÓN ROUND TRIP acabado  en 17.2
Empezando VOCABULARIO
VOCABULARIO acabado  en 5.98
Empezando ORTOGRAFÍA


You shouldn't move a model that is dispatched using accelerate hooks.


ORTOGRAFÍA acabado en 9.88

                        Evaluación de Calidad de Lengua                         

+-----------------+------------------------------------------------------+
| Clave           | Valor                                                |
+-----------------+------------------------------------------------------+
| ttr             | 0.4343505727217461                                   |
| entropy         | 5.896792471339753                                    |
| ngram_overlap   | 0.0                                                  |
| freq_target     | 0.6022653579427514                                   |
| freq_comparison | {'fr': 0.4348357809339771, 'es': 0.9227671416967272} |
| calidad         | 0.14281765002139218                                  |
+-----------------+------------------------------------------------------+

                            Evaluación de Traducción                            

+------+--------------------+
| Clave | Valor            

# Gallego

## Mistral

In [6]:
idioma = "gallego"
modelo = "mistralai/Mistral-7B-Instruct-v0.3"
resultados = evaluate_benchmark(modelo, idioma, os.getenv("HF_TOKEN"), 100)
date = time.localtime(time.time())
filename = f"resultados_{idioma}_{modelo.split('/')[-1]}_{time.strftime('%m-%d_%H-%M-%S', date)}.json"
with open(base + filename, "w") as f:
    json.dump(resultados, f)

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

You set `add_prefix_space`. The tokenizer needs to be converted from the slow tokenizers


...

TRADUCCIÓN ROUND TRIP acabado  en 20.17
Empezando VOCABULARIO


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for o

VOCABULARIO acabado  en 5.88
Empezando ORTOGRAFÍA


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for o

ORTOGRAFÍA acabado en 14.94

                        Evaluación de Calidad de Lengua                         

+-----------------+-------------------------------------------------------+
| Clave           | Valor                                                 |
+-----------------+-------------------------------------------------------+
| ttr             | 0.5359834784442287                                    |
| entropy         | 6.54864711563072                                      |
| ngram_overlap   | 0.0                                                   |
| freq_target     | 0.7194853131384008                                    |
| freq_comparison | {'fr': 0.31575829038658326, 'es': 0.6095678988530958} |
| calidad         | 0.30929109027830637                                   |
+-----------------+-------------------------------------------------------+

                            Evaluación de Traducción                            

+------+--------------------+
| Clave | Valor 

## Salamandra

In [7]:
idioma = "gallego"
modelo = "BSC-LT/salamandra-7b-instruct"
resultados = evaluate_benchmark(modelo, idioma, os.getenv("HF_TOKEN"), 100)
date = time.localtime(time.time())
filename = f"resultados_{idioma}_{modelo.split('/')[-1]}_{time.strftime('%m-%d_%H-%M-%S', date)}.json"
with open(base + filename, "w") as f:
    json.dump(resultados, f)

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/4.81M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/19.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/513 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/730 [00:00<?, ?B/s]

...

TRADUCCIÓN ROUND TRIP acabado  en 29.67
Empezando VOCABULARIO


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for o

VOCABULARIO acabado  en 8.66
Empezando ORTOGRAFÍA


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for o

ORTOGRAFÍA acabado en 12.82

                        Evaluación de Calidad de Lengua                         

+-----------------+-------------------------------------------------------+
| Clave           | Valor                                                 |
+-----------------+-------------------------------------------------------+
| ttr             | 0.7262680258540952                                    |
| entropy         | 8.282843713207543                                     |
| ngram_overlap   | 0.0008733782706385445                                 |
| freq_target     | 0.8381700762867667                                    |
| freq_comparison | {'fr': 0.22633786876580428, 'es': 0.5359004952226759} |
| calidad         | 0.3008585990641629                                    |
+-----------------+-------------------------------------------------------+

                            Evaluación de Traducción                            

+------+--------------------+
| Clave | Valor 

## Gemma

In [8]:
idioma = "gallego"
modelo = "google/gemma-7b-it"
resultados = evaluate_benchmark(modelo, idioma, os.getenv("HF_TOKEN"), 100)
date = time.localtime(time.time())
filename = f"resultados_{idioma}_{modelo.split('/')[-1]}_{time.strftime('%m-%d_%H-%M-%S', date)}.json"
with open(base + filename, "w") as f:
    json.dump(resultados, f)

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Some weights of GemmaForCausalLM were not initialized from the model checkpoint at google/gemma-7b-it and are newly initialized: ['lm_head.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Tokenizer loaded: GemmaTokenizerFast
Model loaded with 4-bit quantization: GemmaForCausalLM
Model device: cuda:0
Empezando CALIDAD DE LENGUA
CALIDAD DE LENGUA acabada en 5.56 minutos
Empezando TRADUCCIÓN DIRECTA
TRADUCCIÓN DIRECTA acabada en 14.88
Empezando TRADUCCIÓN ROUND TRIP
TRADUCCIÓN ROUND TRIP acabado  en 30.72
Empezando VOCABULARIO
VOCABULARIO acabado  en 7.88
Empezando ORTOGRAFÍA


You shouldn't move a model that is dispatched using accelerate hooks.


ORTOGRAFÍA acabado en 15.18

                        Evaluación de Calidad de Lengua                         

+-----------------+------------------------------------------------------------+
| Clave           | Valor                                                      |
+-----------------+------------------------------------------------------------+
| ttr             | 0.15836247327925038                                        |
| entropy         | 3.2982736647129824                                         |
| ngram_overlap   | 0.0                                                        |
| freq_target     | 0.000679325161490719                                       |
| freq_comparison | {'fr': 0.0031571416622474943, 'es': 0.0020663674846565965} |
| calidad         | 0.007860053468631093                                       |
+-----------------+------------------------------------------------------------+

                            Evaluación de Traducción                          

## Qwen

In [4]:
idioma = "gallego"
modelo = "Qwen/Qwen2.5-7B-Instruct"
resultados = evaluate_benchmark(modelo, idioma, os.getenv("HF_TOKEN"), 100)
date = time.localtime(time.time())
filename = f"resultados_{idioma}_{modelo.split('/')[-1]}_{time.strftime('%m-%d_%H-%M-%S', date)}.json"
with open(base + filename, "w") as f:
    json.dump(resultados, f)

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


...

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Tokenizer loaded: Qwen2TokenizerFast
Model loaded with 4-bit quantization: Qwen2ForCausalLM
Model device: cuda:0
Empezando CALIDAD DE LENGUA
CALIDAD DE LENGUA acabada en 3.55 minutos
Empezando TRADUCCIÓN DIRECTA
TRADUCCIÓN DIRECTA acabada en 6.98
Empezando TRADUCCIÓN ROUND TRIP
TRADUCCIÓN ROUND TRIP acabado  en 15.68
Empezando VOCABULARIO
VOCABULARIO acabado  en 4.16
Empezando ORTOGRAFÍA


You shouldn't move a model that is dispatched using accelerate hooks.


ORTOGRAFÍA acabado en 5.74

                        Evaluación de Calidad de Lengua                         

+-----------------+-------------------------------------------------------+
| Clave           | Valor                                                 |
+-----------------+-------------------------------------------------------+
| ttr             | 0.4750400906450662                                    |
| entropy         | 6.715766872548803                                     |
| ngram_overlap   | 0.0034168853497421048                                 |
| freq_target     | 0.8527988167950247                                    |
| freq_comparison | {'es': 0.6825851450452504, 'fr': 0.32543560011016875} |
| calidad         | 0.29426787511501507                                   |
+-----------------+-------------------------------------------------------+

                            Evaluación de Traducción                            

+------+--------------------+
| Clave | Valor  